# cxr-clahe-loso: Colab runner
Run the cells in order. After a disconnect, reconnect and run cells 1, 2 and 4 again: finished runs are skipped and unfinished ones resume.
Runtime > Change runtime type > GPU first.

In [ ]:
# 1. Drive (results are stored here so they survive a disconnect) + code
from google.colab import drive
drive.mount('/content/drive')
import os
REPO_URL = 'https://github.com/Dylansius11/tugas-biomedik.git'  # change if the repo moves
BRANCH = 'main'
if not os.path.exists('/content/repo'):
    !git clone -b $BRANCH $REPO_URL /content/repo
%cd /content/repo
!pip install -q -r requirements.txt && pip install -q -e .

In [ ]:
# 2. Keep results on Drive via symlink
import os, shutil
drive_results = '/content/drive/MyDrive/cxr-clahe-loso/results'
os.makedirs(drive_results, exist_ok=True)
if os.path.isdir('results') and not os.path.islink('results'):
    shutil.rmtree('results')
if not os.path.islink('results'):
    os.symlink(drive_results, 'results')
!ls results

In [ ]:
# 3. Data (only needed once per session). Option A: download from Mendeley (~2 GB)
if not os.path.exists('data/processed/manifest.csv'):
    !bash scripts/download_data.sh
    !python scripts/prepare_data.py
# Option B (faster): put processed.zip from the shared Drive into data/ and run:
# !unzip -q /content/drive/MyDrive/processed.zip -d data/processed
!python -m pytest -q

In [ ]:
# 4. Tuning (skips folds that are done) then all 36 runs (skips finished ones)
for fold in ['ho_darwin', 'ho_montgomery', 'ho_shenzhen']:
    !python -m cxrseg.tune_clahe --fold $fold
!python scripts/run_all.py

In [ ]:
# 5. Progress
!python scripts/check_runs.py